# Paper 2C — employee attrition, honestly evaluated

The HR-facing analysis: what the six employee-turnover datasets look like under each of the
three protocols, and what an HR analytics team should conclude.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

CODE = Path.cwd()
while CODE.name != 'Code' and CODE.parent != CODE:
    CODE = CODE.parent
sys.path.insert(0, str(CODE))

import numpy as np, pandas as pd
import leakage_lib as L
pd.set_option('display.width', 200, 'display.max_columns', 60)
print('corpus  :', L.PROC)
print('results :', L.RES)

R = pd.read_csv(L.RES/'raw_results.csv')
if 'error' in R.columns:
    R = R[R.error.isna()]
man = L.manifest()
piv = L.paired(R, man)
H = piv[piv.domain=='HR']
print(f'{H.dataset.nunique()} HR datasets, {len(H):,} matched comparisons')

## The datasets

In [ ]:
man[man.domain=='HR'][['name','n','n_features','n_minority','imbalance_ratio','source']]

## How much the protocol is worth

In [ ]:
print(f'correct  {H.correct.mean():.4f}')
print(f'leaked   {H.leaked.mean():.4f}')
print(f'premium  {H.premium.mean():.4f}')
pd.read_csv(L.RES/'table8_hr.csv').round(4)

In [ ]:
from IPython.display import Image, display
display(Image(str(L.FIG/'fig06_hr.png')))
display(Image(str(L.FIG/'fig07_hr_vs_benchmark.png')))

## Protocol versus algorithm

The comparison that matters for the paper's claim: moving the resampler across the split
changes the reported score by more than switching between any two model families.

In [ ]:
spread_protocol  = H.leaked.mean() - H.correct.mean()
byclf = H.groupby('clf')[['correct']].mean()
spread_algorithm = float(byclf.correct.max() - byclf.correct.min())
print(f'protocol effect  {spread_protocol:.4f}')
print(f'algorithm spread {spread_algorithm:.4f}  (best minus worst family, correct protocol)')
byclf.round(4)

## Is resampling worth doing at all?

In [ ]:
none = R[(R.condition=='none') & (R.dataset.isin(H.dataset.unique()))]
n_ = none.groupby(['dataset','clf','seed']).f1.mean()
c_ = H.groupby(['dataset','clf','seed']).correct.mean()
m = pd.concat({'none': n_, 'correct': c_}, axis=1).dropna()
print(f'no-resampling beats correctly-ordered resampling in {(m.none>m.correct).mean():.1%} '
      f'of matched HR comparisons')
m.groupby(level='dataset').mean().round(4)

## Build the manuscript

In [ ]:
!python "/root/p2/Code/Paper2C/build_paper2C.py"